## Generates the synthetic Data for Training the Model

First we need to do our imports and get our master csv file.

In [18]:
import pandas as pd
import nvdlib
import pprint
import json
import os
from dotenv import load_dotenv
import urllib.parse

In [6]:
master_df = pd.read_csv('normal_master.csv')
master_df.head()

,model_name,vendor,device_type,keyword_lookup
0,Xbox Series X,Microsoft,Gaming Console,Microsoft Xbox
1,Nintendo Switch,Nintendo,Gaming Console,Nintendo Switch
2,PlayStation 5,Sony,Gaming Console,Sony PlayStation
3,Plex Media Server,Plex,Media Server,Plex Plex
4,Lockerstor (ADM),Asustor,NAS,Asustor Lockerstor


Now we connect to the NVD API.

In [7]:
load_dotenv()

nvd_api_key = os.getenv('NVD_API_KEY')

In [10]:
request = nvdlib.searchCVE(cveId='CVE-2021-26855', key=nvd_api_key, delay=2)
if request:
    pprint.pprint(request)

[{'id': 'CVE-2021-26855', 'sourceIdentifier': 'secure@microsoft.com', 'published': '2021-03-03T00:15:12.103', 'lastModified': '2025-10-30T19:55:58.670', 'vulnStatus': 'Analyzed', 'cveTags': [], 'descriptions': [{'lang': 'en', 'value': 'Microsoft Exchange Server Remote Code Execution Vulnerability'}, {'lang': 'es', 'value': 'Una Vulnerabilidad de Ejecución de código remota de Microsoft Exchange Server. Este ID de CVE es diferente de CVE-2021-26412, CVE-2021-26854, CVE-2021-26857, CVE-2021-26858, CVE-2021-27065, CVE-2021-27078'}], 'metrics': {'cvssMetricV31': [{'source': 'secure@microsoft.com', 'type': 'Secondary', 'cvssData': {'version': '3.1', 'vectorString': 'CVSS:3.1/AV:N/AC:L/PR:N/UI:N/S:U/C:H/I:H/A:N', 'baseScore': 9.1, 'baseSeverity': 'CRITICAL', 'attackVector': 'NETWORK', 'attackComplexity': 'LOW', 'privilegesRequired': 'NONE', 'userInteraction': 'NONE', 'scope': 'UNCHANGED', 'confidentialityImpact': 'HIGH', 'integrityImpact': 'HIGH', 'availabilityImpact': 'NONE'}, 'exploitabilit

In [11]:
device_name = "Ring Video Doorbell"

# Search for CVEs containing the device name
# limit=25 restricts the results to the top 25 matches
results = nvdlib.searchCVE(keywordSearch=device_name, limit=25, key=nvd_api_key, delay=2)

for entry in results:
    print(entry)

{'id': 'CVE-2015-4400', 'sourceIdentifier': 'cve@mitre.org', 'published': '2018-02-06T16:29:00.527', 'lastModified': '2024-11-21T02:31:00.143', 'vulnStatus': 'Modified', 'cveTags': [], 'descriptions': [{'lang': 'en', 'value': 'Ring (formerly DoorBot) video doorbells allow remote attackers to obtain sensitive information about the wireless network configuration by pressing the set up button and leveraging an API in the GainSpan Wi-Fi module.'}, {'lang': 'es', 'value': 'Los videoporteros Ring (anteriormente DoorBot) permiten que atacantes remotos obtengan información sensible sobre la configuración de red inalámbrica presionando el botón de configuración y utilizando una API en el módulo Wi-Fi GainSpan.'}], 'metrics': {'cvssMetricV30': [{'source': 'nvd@nist.gov', 'type': 'Primary', 'cvssData': {'version': '3.0', 'vectorString': 'CVSS:3.0/AV:P/AC:L/PR:N/UI:N/S:U/C:H/I:N/A:N', 'baseScore': 4.6, 'baseSeverity': 'MEDIUM', 'attackVector': 'PHYSICAL', 'attackComplexity': 'LOW', 'privilegesRequ

Now we will get all CVEs for the devices we are interested in and store them in a json file. Limit to top 25 CVEs returned by the API.

In [20]:
def extract_cve_context(nvd_data):
    """
    Parses a raw NVD API response to extract high-value context
    for LLM remediation analysis.
    """
    
    # 1. Extract ID and Description (English)
    cve_id = nvd_data.id
    
    # distinct descriptions often exist for different languages
    descriptions = nvd_data.descriptions
    description_text = next(
        (item.value for item in descriptions if item.lang == 'en'), 
        "No description available."
    )

    # 2. Extract CVSS Vector (The Technical "DNA" of the attack)
    # We prioritize V3.1 > V3.0 > V2.0 because V3 is more descriptive regarding physical/local vectors.
    vector_string = "N/A"
    
    try: 
        if nvd_data.v31vector:
            vector_string = nvd_data.v31vector
        elif nvd_data.v2vector:
            vector_string = nvd_data.v2vector
    except AttributeError:
        print("No CVSS vector found.")
        vector_string = "N/A"

    # 3. Extract Reference URLs (Search Seeds)
    # We grab all URLs to give the LLM maximum surface area for search.
    refs = nvd_data.references
    reference_urls = [r.url for r in refs]

    try:
        # 4. Extract Weakness Type (CWE)
        # Useful for finding general best practices if no specific patch exists.
        cwe_list = nvd_data.cwe
        cwe_ids = [cwe for cwe in cwe_list] if cwe_list else []
    except AttributeError:
        print("No CWE information found.")
        cwe_ids = []
    
    try:
        # 5. Extract Affected CPEs (Software/Hardware Identities)
        configurations = nvd_data.configurations
        affected_cpes = []

        for config in configurations:
            for node in config.nodes:    
                for match in node.cpeMatch:
                    if match.vulnerable:
                        affected_cpes.append(match.criteria)
    except AttributeError:
        print("No affected CPEs found.")
        affected_cpes = []

    # Build the structured context object
    return {
        "cve_id": cve_id,
        "description": description_text,
        "cvss_vector": vector_string,
        "cwe_ids": list(set(cwe_ids)), # Remove duplicates
        "affected_components": list(set(affected_cpes)),
        "reference_links": reference_urls
    }

In [ ]:
discovered_cves = {}

for index, row in master_df.iterrows():
    keyword = row["keyword_lookup"]

    current_finding = []

    encoded_keyword = urllib.parse.quote(keyword)
    results = nvdlib.searchCVE(keywordSearch=encoded_keyword, limit=25, key=nvd_api_key, delay=1)
    for entry in results:
        print(entry)
        data = extract_cve_context(entry)
        current_finding.append(data)

    discovered_cves[keyword] = current_finding

with open('discovered_cves.json', 'w') as f:
    json.dump(discovered_cves, f, indent=4)

{'id': 'CVE-2007-1220', 'sourceIdentifier': 'cve@mitre.org', 'published': '2007-03-02T22:19:00.000', 'lastModified': '2025-04-09T00:30:58.490', 'vulnStatus': 'Deferred', 'cveTags': [], 'descriptions': [{'lang': 'en', 'value': 'The Hypervisor in Microsoft Xbox 360 kernel 4532 and 4548 does not properly verify the parameters passed to the syscall dispatcher, which allows attackers with physical access to bypass code-signing requirements and execute arbitrary code.'}, {'lang': 'es', 'value': 'El Hypervisor en Microsoft Xbox 360 kernel 4532 y 4548 no verifica correctamente los parámetros pasados al despachador del syscall, que permite que los atacantes con acceso físico evitar requisitos la firma de código y que ejecuten código de su elección.'}], 'metrics': {'cvssMetricV2': [{'source': 'nvd@nist.gov', 'type': 'Primary', 'cvssData': {'version': '2.0', 'vectorString': 'AV:L/AC:H/Au:N/C:C/I:C/A:C', 'baseScore': 6.2, 'accessVector': 'LOCAL', 'accessComplexity': 'HIGH', 'authentication': 'NONE